In [1]:
!pip install pypdf chromadb langchain langchain-community langchain-huggingface sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [3]:
from pypdf import PdfReader

pdf = PdfReader("bula_1782399219881.pdf")

texto = ""

for pagina in pdf.pages:
    texto += str(pagina.extract_text())

print("Tamanho do texto:", len(texto))

Tamanho do texto: 18022


In [4]:
print(texto[:1000])

1 Bula_Paciente 
ibuprofeno 
Vitamedic Indúst
ria Farmacêutica Ltda 
Suspensão Gotas 
100mg/mL 2 Bula_Paciente 
ibuprofeno 
Medicamento genérico, Lei nº 9.787, de 1999 
I – IDENTIFICAÇÃO DO MEDICAMENTO: 
APRESENTAÇÕES 
Suspensão Gotas.  
Embalagens com: 1, 50, 100 ou 200 frascos com 20mL. 
USO ORAL  
USO ADULTO E PEDIÁTRICO ACIMA DE 6 MESES  
COMPOSIÇÃO 
Cada mL da suspensão gotas* contém: 
ibuprofeno......................................................................................................................................................................100mg 
veículo q.s.p.......................................................................................................................................................................1mL 
Excipientes: goma xantana, glicerol, benzoato de sódio, propilenoglicol, ácido cítrico, aroma tutti-frutti, sorbitol, sacarina 
sódica dihidratada, ciclamato de sódio, dióxido de titânio, po lissorbato 80, essência de baunilha hidrossolúv

In [11]:
chunks = []

chunks.append("""
IDENTIFICAÇÃO DO MEDICAMENTO
Ibuprofeno 100mg
Suspensão oral
""")

chunks.append("""
PARA QUE SERVE
O ibuprofeno é usado para dor e febre.
""")

chunks.append("""
COMO FUNCIONA
Ação entre 15 e 30 minutos.
""")

chunks.append("""
QUANDO NÃO USAR
Alergia ao ibuprofeno ou menores de 6 meses.
""")

chunks.append("""
POSOLOGIA

Adultos: usar conforme orientação médica.
Crianças: seguir recomendação médica.
""")

chunks.append("""
EFEITOS COLATERAIS

Pode causar náusea, tontura e desconforto abdominal.
""")

chunks.append("""
INTERAÇÕES MEDICAMENTOSAS

Evitar uso concomitante com álcool.
""")

chunks.append("""
GRAVIDEZ E AMAMENTAÇÃO

Contraindicado no terceiro trimestre da gravidez.
""")

print("Chunks:", len(chunks))


Chunks: 8


In [17]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [18]:
print(len(chunks))

8


In [19]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [20]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

In [21]:
query = "para que serve ibuprofeno?"

results = db.similarity_search(query, k=2)

for r in results:
    print("\n--- RESULTADO ---\n")
    print(r.page_content)


--- RESULTADO ---


PARA QUE SERVE
O ibuprofeno é usado para dor e febre.


--- RESULTADO ---


PARA QUE SERVE
O ibuprofeno é usado para dor e febre.



In [22]:
def buscar_resposta(pergunta):

    resultados = db.similarity_search(pergunta, k=2)

    resposta = ""

    for doc in resultados:
        resposta += doc.page_content + "\n\n"

    return f"Segundo a bula:\n\n{resposta}"

In [23]:
print(buscar_resposta("para que serve ibuprofeno"))

Segundo a bula:


PARA QUE SERVE
O ibuprofeno é usado para dor e febre.



PARA QUE SERVE
O ibuprofeno é usado para dor e febre.





In [ ]:
while True:

    pergunta = input("Pergunta: ")

    if pergunta.lower() == "sair":
        break

    print(buscar_resposta(pergunta))

Pergunta: efeitos colaterais
Segundo a bula:


EFEITOS COLATERAIS

Pode causar náusea, tontura e desconforto abdominal.



GRAVIDEZ E AMAMENTAÇÃO

Contraindicado no terceiro trimestre da gravidez.



Pergunta: indicação 
Segundo a bula:


GRAVIDEZ E AMAMENTAÇÃO

Contraindicado no terceiro trimestre da gravidez.



INTERAÇÕES MEDICAMENTOSAS

Evitar uso concomitante com álcool.



